In [1]:
import requests
from lxml import etree
import requests
import pandas as pd
import os
import re
import datetime
def create_dateframe(html):
    digtal = re.compile('[0-9]+')
    text_list = []
    tr_list = html.xpath('//*[@id="pl_top_realtimehot"]/table/tbody/tr')
    data = pd.DataFrame(columns = ['排行','事件','热度','链接','事件拼接链接'])
    for tr in tr_list:
        tr_rank = tr.xpath('./td[1]/text()')
        if tr_rank == []:
            tr_rank = '上升'
            tr_text = tr.xpath('./td[2]/a/text()')[0]
            tr_heat = ''
            tr_href = 'https://s.weibo.com' +tr.xpath('./td[2]/a/@href')[0]
            tr_splice = tr_rank+'.'+ tr_text +' '+ tr_href
        else:
            tr_rank = tr_rank[0]
            tr_text = tr.xpath('./td[2]/a/text()')[0]
            tr_heat = tr.xpath('./td[2]/span/text()')[0]
            tr_heat = digtal.findall(tr_heat)
            tr_splice = tr_rank+'.'+ tr_text+' '+ tr_href
            if tr_heat != []:
                tr_heat = tr_heat[0]
            else:
                tr_heat = ''
            tr_href = 'https://s.weibo.com' + tr.xpath('./td[2]/a/@href')[0]
        data = pd.concat([data,pd.DataFrame([tr_rank,tr_text,tr_heat,tr_href,tr_splice],index = ['排行','事件','热度','链接','事件拼接链接']).T])
        data = data.reset_index(drop=True)
    return data
def save_excel(df,time_stamp):
    time_now = time_stamp.strftime('%Y.%m.%d')
    os.chdir("C:\\Users\\86139\\Desktop\\保存热搜数据")
    file_name = time_now+'微博热搜的数据.xlsx'
    df.to_excel(file_name,index = False)
    print(time_now,'数据更新成功')

In [2]:
url = 'https://s.weibo.com/top/summary'
headers = {
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; WOW64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/104.0.0.0 Safari/537.36',
    'referer': 'https://passport.weibo.com/',
    'cookie': 'SUB=_2AkMS-WPBf8NxqwFRmfoRxGzibo9zyQvEieKkpZIaJRMxHRl-yT9kqkxdtRB6OXlNLnasnhc3GMLU1lxeZgVSuqQcvtNF; SUBP=0033WrSXqPxfM72-Ws9jqgMF55529P9D9WW.zgb4bVoLSbXXhE74YvNV; _s_tentry=passport.weibo.com; Apache=4906935630120.408.1705372918261; SINAGLOBAL=4906935630120.408.1705372918261; ULV=1705372918271:1:1:1:4906935630120.408.1705372918261:'
}
time_stamp = datetime.datetime.now()
response = requests.get(url = url,headers = headers)
response.raise_for_status()
response.encoding = response.apparent_encoding
html_1 = etree.HTML(response.text)
data = create_dateframe(html_1)
save_excel(data,time_stamp)

2024.02.04 数据更新成功
